In [1]:
import torch

In [2]:
from tangermeme.tools import fimo

# import sys
# sys.path.insert(0, "/home1/smaruj/tangermeme")  # Add the directory where "ledidi" is located
# from tangermeme.tools import fimo

In [ ]:
# device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# print(device)

In [3]:
# original seq, no modifications
X_slice = torch.load("/scratch1/smaruj/ledidi_targets/constant_boundary_background/X_no_mod.pt", weights_only=True) #, map_location=device)

In [4]:
# edited sequence, with modifications
X_hat_slice = torch.load("/scratch1/smaruj/ledidi_targets/constant_boundary_background/X_with_mod.pt", weights_only=True) #, map_location=device)

In [5]:
X_slice_bin = X_slice[:,:,4096:-4096]

In [6]:
X_hat_slice_bin = X_hat_slice[:,:,4096:-4096]

In [7]:
CTCF_PWM = "/home1/smaruj/IterativeMutagenesis/MA0139.1.meme"

In [8]:
import numpy as np

def read_meme_pwm_as_numpy(filename):
    pwm_list = []  # List to store PWM rows
    
    with open(filename, 'r') as file:
        in_matrix_section = False
        
        for line in file:
            line = line.strip()
            
            # Check if we are reading the PWM matrix
            if line.startswith("letter-probability matrix"):
                in_matrix_section = True  # Start reading matrix data
                continue  # Skip this header line
            
            # If we are in the matrix section, process the rows
            if in_matrix_section and line:
                pwm_row = [float(value) for value in line.split()]  # Parse values
                pwm_list.append(pwm_row)  # Append to the PWM list
            
            # If we encounter a new MOTIF or the end of file, stop matrix reading
            if line.startswith("MOTIF") and in_matrix_section:
                break
    
    # Convert the list to a numpy array
    pwm_array = np.array(pwm_list).T # Transpose to get shape (4, N)
    
    return pwm_array

In [9]:
pwm_CTCF = read_meme_pwm_as_numpy(CTCF_PWM)

In [10]:
pwm_CTCF_tensor = torch.from_numpy(pwm_CTCF).float()

In [11]:
pwm_CTCF_tensor

tensor([[0.0953, 0.1829, 0.3078, 0.0613, 0.0088, 0.8149, 0.0438, 0.1173, 0.9331,
         0.0055, 0.3655, 0.0593, 0.0132, 0.0615, 0.1144, 0.4092, 0.0903, 0.1289,
         0.4427],
        [0.3187, 0.1588, 0.0537, 0.8762, 0.9890, 0.0142, 0.5783, 0.4748, 0.0121,
         0.0000, 0.0033, 0.0132, 0.0000, 0.0088, 0.8064, 0.0143, 0.5308, 0.3546,
         0.1993],
        [0.0832, 0.4534, 0.4918, 0.0230, 0.0000, 0.0712, 0.3658, 0.0526, 0.0351,
         0.9912, 0.6213, 0.5532, 0.9780, 0.8516, 0.0055, 0.5578, 0.3381, 0.0804,
         0.2930],
        [0.5027, 0.2048, 0.1468, 0.0394, 0.0022, 0.0997, 0.0120, 0.3553, 0.0197,
         0.0033, 0.0099, 0.3743, 0.0088, 0.0780, 0.0737, 0.0187, 0.0407, 0.4361,
         0.0650]])

In [12]:
motifs_dict = {"CTCF": pwm_CTCF_tensor}

In [13]:
X_slice_bin_cpu = X_slice_bin.cpu().detach().numpy()

In [14]:
hits = fimo.fimo(motifs=motifs_dict, sequences=X_slice_bin_cpu, threshold=1e-4, reverse_complement=True)

In [15]:
hits

[Empty DataFrame
 Columns: [motif_name, motif_idx, sequence_name, start, end, strand, score, p-value]
 Index: []]

In [16]:
X_hat_slice_bin_cpu = X_hat_slice_bin.cpu().detach().numpy()

In [17]:
hits_hat = fimo.fimo(motifs=motifs_dict, sequences=X_hat_slice_bin_cpu, threshold=1e-4, reverse_complement=True)

In [20]:
hits_hat[0]["score"].sum()

259.2137573957443

In [ ]:
CTCF_hits = hits_hat[0]

In [ ]:
CTCF_hits[CTCF_hits["strand"] == "+"]["score"]